In [1]:
import os

import numpy as np
import pandas as pd

import torch
from torch import nn

/tmp/ipykernel_994007/1099783377.py:4: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
import sys, os
sclembas = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas))
from scLEMBAS import parse_network, io
from scLEMBAS.model.scl import SignalingModel
from scLEMBAS.model.tfa import TFA

In [3]:
n_cores = 12
os.environ["OMP_NUM_THREADS"] = str(n_cores)
os.environ["MKL_NUM_THREADS"] = str(n_cores)
os.environ["OPENBLAS_NUM_THREADS"] = str(n_cores)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(n_cores)
os.environ["NUMEXPR_NUM_THREADS"] = str(n_cores)

seed = 888

device = "cuda" if torch.cuda.is_available() else "cpu"

data_path = '/nobackup/users/hmbaghda/scLEMBAS/analysis'
models_path = os.path.join(data_path, 'processed', 'models')
if not os.path.isdir(models_path):
    os.mkdir(models_path)

In [4]:
tf_adata = io.read_tfad(file_name = os.path.join(data_path, 'processed', 'ID_tf_activity.h5ad'))

In [5]:
source_label = 'source_genesymbol'
target_label = 'target_genesymbol'
weight_label = 'mode_of_action'
stimulation_label = 'consensus_stimulation'
inhibition_label = 'consensus_inhibition'

sn_ppis = parse_network.load_network('omnipath', organism = 'mouse', static = True)
sn_ppis = parse_network.extract_network(sn_ppis, curation_effort_thresh = 5, n_references_thresh = 3,
                                        resources = ['HuRI','IntAct','KEGG-MEDICUS','NetPath','Reactome_SignaLink3','SPIKE','SignaLink3','SIGNOR', 
                                                'Baccin2019', 'Ramilowski2015', 'Reactome_LRdb', 'UniProt_LRdb', 'CellChatDB', 'CellPhoneDB', 'connectomeDB2020', 'scConnect'], 
                                        drop_self = True, verbose = True)

tf_labels = tf_adata.var.index.unique().tolist()

ligand_labels = tf_adata.obs['sample'].unique().tolist()
ligand_labels = [(l[0] + l[1:].lower()).replace('-', '') for l in ligand_labels] # mouse naming convention

# filter for paths b/w ligand and tf
fn_1 = parse_network.fully_connected_network(sn_ppis, ligand_labels, tf_labels, source_label = source_label, target_label = target_label, 
                       path_finder = 'shortest')
fn_2 = parse_network.fully_connected_network(sn_ppis, ligand_labels, tf_labels, source_label = source_label, target_label = target_label, 
                       path_finder = 'connected')
# of the methods to identify paths, retain the one that has the most interactions
if fn_1.shape[0] > fn_2.shape[0]:
    sn_ppis = fn_1
else:
    sn_ppis = fn_2

del fn_1, fn_2

sn_ppis.loc[sn_ppis[(sn_ppis[stimulation_label] == 1) & (sn_ppis[inhibition_label] == 1)].index, 
    [stimulation_label, inhibition_label]] = [False, False]
sn_ppis = parse_network.format_network(sn_ppis, weight_label, stimulation_label, inhibition_label)

all_nodes = sn_ppis[source_label].tolist() + sn_ppis[target_label].tolist()
input_ligands_available = sorted(set(ligand_labels).intersection(all_nodes))

The thresholds filtered 21403  of 28277 interactions
The resources filtered 937  of 6874 interactions


100%|████████████████████████████████████| 8122/8122 [00:00<00:00, 21322.27it/s]


In [6]:
subset_tf = tf_adata[tf_adata.obs.TF_clusters.isin(['9', '15'])]
sample_size = subset_tf.obs.TF_clusters.value_counts().min()
large_cluster = subset_tf.obs.TF_clusters.value_counts().idxmax()
small_cluster = subset_tf.obs.TF_clusters.value_counts().idxmin()
large_cluster_barcodes = subset_tf.obs[subset_tf.obs.TF_clusters == large_cluster].index
small_cluster_barcodes = subset_tf.obs[subset_tf.obs.TF_clusters == small_cluster].index.tolist()
np.random.seed(seed)
lcb_sub = list(np.random.choice(large_cluster_barcodes, sample_size, replace = False))
subset_tf = subset_tf[lcb_sub + small_cluster_barcodes, :]
subset_tf.obs.TF_clusters.value_counts()

np.random.seed(seed)
selected_ligand = np.random.choice(input_ligands_available, 1)[0]

ligand_input = pd.DataFrame(subset_tf.obs.TF_clusters.map({'9': 0, '15': 1}))
ligand_input.columns = [selected_ligand]
tf_output = pd.DataFrame(subset_tf.X, index = subset_tf.obs.index, columns = subset_tf.var.index)

In [7]:
# linear scaling of inputs/outputs
projection_amplitude_in = 3
projection_amplitude_out = 1.2
# other parameters
bionet_params = {'target_steps': 100, 'max_steps': 120, 'exp_factor':50, 'tolerance': 1e-5, 'leak':1e-2} # fed directly to model

# training parameters
lr_params = {'max_iter': 5000, 
             'learning_rate': 2e-3}
other_params = {'batch_size': 256, 'noise_level': 10, 'gradient_noise_level': 1e-9}
regularization_params = {'param_lambda_L2': 1e-6, 'moa_lambda_L1': 0.1, 'ligand_lambda_L2': 1e-5, 'uniform_lambda_L2': 1e-4, 
                   'uniform_max': 1/projection_amplitude_out, 'spectral_loss_factor': 1e-5}
spectral_radius_params = {'n_probes_spectral': 5, 'power_steps_spectral': 50, 'subset_n_spectral': 10}
target_spectral_radius = 0.8
hyper_params = {**lr_params, **other_params, **regularization_params, **spectral_radius_params}

# TFA
encoder_dist = None
decoder_dist = 'gauss'

e_hp = {'batch_momentum': 0.01, 'layer_norm': False, 'dropout_rate': 0.1, 
         'activation_fn': torch.nn.ReLU, 'n_hidden_nodes': [64], 'linear_output': True}
d_hp = {'batch_momentum': 0.01, 'layer_norm': False, 'dropout_rate': 0.1, 
         'activation_fn': torch.nn.ReLU, 'n_hidden_nodes': [64], 'linear_output': True}
ve_hp = {'batch_momentum': 0.01, 'layer_norm': False, 'dropout_rate': 0.1, 
         'activation_fn': nn.ReLU, 'n_hidden_nodes': [64], 'var_min': 1e-4}
vd_hp = {'batch_momentum': 0.01, 'layer_norm': False, 'dropout_rate': 0.1, 
         'activation_fn': nn.ReLU, 'n_hidden_nodes': [64], 'var_min': 1e-4}

Hyper_params = {
    'tfa': {'n_latent': 32, 'cat_max_norm': 1, 'encoder_dist': encoder_dist, 'decoder_dist': decoder_dist}, 
    'encoder': ve_hp if encoder_dist == 'guass' else e_hp,
    'decoder': vd_hp if decoder_dist == 'guass' else d_hp,
    'discriminator': {'batch_momentum': 0.01, 'layer_norm': False, 'dropout_rate': 0.1, 'activation_fn': nn.ReLU, 'n_hidden_nodes': [16, 16, 16], 'return_logits': True}

}

In [8]:
mod = SignalingModel(net = sn_ppis,
                     X_in = ligand_input,
                     y_out = tf_output, 
                     activation_function = 'MML', 
                     skip_bionet_out = False,
                     covariates = subset_tf.obs,
                     categorical_covariate_keys = ['TF_clusters', 'celltype'],
                     projection_amplitude_in = projection_amplitude_in, projection_amplitude_out = projection_amplitude_out,
                     weight_label = weight_label, source_label = source_label, target_label = target_label,
                     bionet_params = bionet_params, 
                     tfa_hyper_params = Hyper_params['tfa'], 
                     encoder_hyper_params = Hyper_params['encoder'],
                     decoder_hyper_params = Hyper_params['decoder'],
                     dtype = torch.float32, device = device, seed = seed)

# Test

In [9]:
# class TestClass:
#     def __init__():
#         pass
# self = TestClass

In [10]:
X_in = mod.df_to_tensor(mod.X_in)[:64, :]
X_full = mod.input_layer(X_in) # input ligands to signaling network
Y_full = mod.signaling_network(X_full) # RNN of full signaling network
Y_hat = mod.output_layer(Y_full) # TF outputs of signaling network

self = mod.tf_autoencoder
z_m, z_v, z_basal = None, None, self.encoder(Y_hat)

Step 1: forward pass combination of covar embeddings and basal:


In [74]:
mod.signaling_network

BioNet()

In [14]:
categories = {'TF_clusters': ['15', '15'], 
             'celltype': ['B_cell', 'cDC1']}

In [58]:
sample_ids = list(self.covariates.index[:64])

In [63]:
categories[cvk]

AGGCCACCAAGTTTGC-35     T_cell_gd
ATCTTCAAGAGCTTTC-37          Treg
GGTGAAGCAGCAGGAT-02    T_cell_CD8
CAGCCAGCAGCCATTA-11    T_cell_CD4
AGACACTAGTCGAAAT-28         MigDC
                          ...    
CTCGAGGGTTCGGTTA-13    T_cell_CD4
CTAGGTATCCGGCAAC-10    T_cell_CD4
GTGATGTCAAATGCTC-23    T_cell_CD8
TTCTCTCCAGGCTCTG-14    T_cell_CD4
CTCCCAAGTGACGCCT-23    T_cell_CD8
Name: celltype, Length: 64, dtype: category
Categories (12, object): ['B_cell', 'Macrophage', 'MigDC', 'Monocyte', ..., 'T_cell_gd', 'Treg', 'cDC1', 'pDC']

In [ ]:
categories = {}
for cat in self.covariates.columns:
    categories[cvk] = self.covariates.loc[sample_ids, cvk]

In [19]:
cat = 'TF_clusters'
labels = categories[cat]

In [72]:
z

tensor([[ 0.0278,  0.0760,  0.1224,  ...,  0.0105, -0.0553, -0.0144],
        [ 0.0278,  0.0760,  0.1224,  ...,  0.0105, -0.0553, -0.0144],
        [ 0.0278,  0.0760,  0.1224,  ...,  0.0105, -0.0553, -0.0144],
        ...,
        [ 0.0278,  0.0760,  0.1224,  ...,  0.0105, -0.0553, -0.0144],
        [ 0.0278,  0.0760,  0.1224,  ...,  0.0105, -0.0553, -0.0144],
        [ 0.0278,  0.0760,  0.1224,  ...,  0.0105, -0.0553, -0.0144]],
       grad_fn=<CloneBackward0>)

In [35]:
self.one_hot[cat][label_idx]

tensor([1, 0])

In [12]:
torch.matmul(self.cat_embeddings)

tensor([[0.0027, 0.1746, 0.0022,  ..., 0.0016, 0.8949, 0.0015],
        [0.0027, 0.1746, 0.0022,  ..., 0.0016, 0.8949, 0.0015],
        [0.0027, 0.1746, 0.0022,  ..., 0.0016, 0.8949, 0.0015],
        ...,
        [0.0027, 0.1746, 0.0022,  ..., 0.0016, 0.8949, 0.0015],
        [0.0027, 0.1746, 0.0022,  ..., 0.0016, 0.8949, 0.0015],
        [0.0027, 0.1746, 0.0022,  ..., 0.0016, 0.8949, 0.0015]],
       grad_fn=<MulBackward0>)

In [11]:
z_basal

tensor([[ 0.0278,  0.0760,  0.1224,  ...,  0.0105, -0.0553, -0.0144],
        [ 0.0278,  0.0760,  0.1224,  ...,  0.0105, -0.0553, -0.0144],
        [ 0.0278,  0.0760,  0.1224,  ...,  0.0105, -0.0553, -0.0144],
        ...,
        [ 0.0278,  0.0760,  0.1224,  ...,  0.0105, -0.0553, -0.0144],
        [ 0.0278,  0.0760,  0.1224,  ...,  0.0105, -0.0553, -0.0144],
        [ 0.0278,  0.0760,  0.1224,  ...,  0.0105, -0.0553, -0.0144]],
       grad_fn=<AddmmBackward0>)

In [37]:
import torch.nn.functional as F

In [11]:
self.cat_mapper

{'TF_clusters': {'15': 0, '9': 1},
 'celltype': {'B_cell': 0,
  'Macrophage': 1,
  'MigDC': 2,
  'Monocyte': 3,
  'NK_cell': 4,
  'Neutrophil': 5,
  'T_cell_CD4': 6,
  'T_cell_CD8': 7,
  'T_cell_gd': 8,
  'Treg': 9,
  'cDC1': 10,
  'pDC': 11}}

In [14]:
self.cat_mapper

{'TF_clusters': {'15': 0, '9': 1},
 'celltype': {'B_cell': 0,
  'Macrophage': 1,
  'MigDC': 2,
  'Monocyte': 3,
  'NK_cell': 4,
  'Neutrophil': 5,
  'T_cell_CD4': 6,
  'T_cell_CD8': 7,
  'T_cell_gd': 8,
  'Treg': 9,
  'cDC1': 10,
  'pDC': 11}}

In [15]:
self.cat_embeddings

ModuleDict(
  (TF_clusters): Embedding(2, 32, max_norm=1)
  (celltype): Embedding(12, 32, max_norm=1)
)

In [ ]:
# do this:
z_cov = (embedding * one-hot) # samples * latent space
z_basal + z_cov 

In [ ]:
embedding = self.cat_embeddings[cat]

In [ ]:
.view(-1,)

12

Step 2: discriminator

https://github.com/theislab/cpa/blob/main/cpa/_task.py#L191